# G-I-A Analytical Lens: Qualitative Walkthrough

This notebook provides an illustrative walkthrough of the **Grounding-Instructibility-Alignment (G-I-A)** analytical lens used in our survey paper *"Neuro-Symbolic AI for Cybersecurity: State of the Art, Challenges, and Opportunities"*.

**Important:** G-I-A supports structured qualitative examination; it is not a validated metric, computable scoring algorithm, ranking method, or benchmark.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.dpi': 150
})

def locate_repo_root():
    """Locate the repository whether the notebook starts in root or notebooks/."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'data' / 'paper_catalog.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate data/paper_catalog.csv')

REPO_ROOT = locate_repo_root()

## 1. G-I-A Dimensions and Schematic Formulations

The three dimensions are described below using the manuscript's schematic formulations. These expressions organize design considerations; they are neither universal training objectives nor validated evaluation metrics.

**Grounding Quality** measures how well the system connects outputs to cybersecurity concepts:

$$\mathcal{G}(\theta, \mathcal{K}) = \frac{1}{|\mathcal{Z}|} \sum_{c \in \mathcal{Z}} \text{Consistency}(\Phi_\theta(x_c), \Psi_{\mathcal{K}}(x_c, c))$$

**Instructibility** describes responsiveness to analyst feedback:

$$\mathcal{I}(\theta, \mathcal{K}, \mathcal{H}) = \mathbb{E}_{h \in \mathcal{H}} \left[ \text{Adaptation}(\Delta\theta_h, \Delta\mathcal{K}_h) \right]$$

**Alignment** describes consistency with organizational objectives:

$$\mathcal{A}(\theta, \mathcal{K}, \mathcal{O}) = \sum_{o \in \mathcal{O}} w_o \cdot \text{Objective}(\Phi_\theta, \Psi_{\mathcal{K}}, o)$$

In [ ]:
DIMENSIONS = ('Grounding', 'Instructibility', 'Alignment')
QUALITATIVE_LEVELS = {'Strong', 'Moderate', 'Limited'}

def qualitative_assessment_record(system, tier, assessments, evidence):
    """Create an inspectable qualitative record without aggregating assessments."""
    missing = set(DIMENSIONS) - set(assessments)
    invalid = set(assessments.values()) - QUALITATIVE_LEVELS
    if missing:
        raise ValueError(f'Missing dimensions: {sorted(missing)}')
    if invalid:
        raise ValueError(f'Unsupported qualitative labels: {sorted(invalid)}')

    return pd.DataFrame([
        {
            'system': system,
            'tier': tier,
            'dimension': dimension,
            'assessment': assessments[dimension],
            'supporting_evidence': evidence[dimension],
        }
        for dimension in DIMENSIONS
    ])

## 2. Illustrative Qualitative Assessment Record

The hypothetical example below shows how each categorical assessment can be paired with a concise evidence statement. It illustrates documentation structure only and is not part of the survey corpus.

In [ ]:
hypothetical_assessments = {
    'Grounding': 'Strong',
    'Instructibility': 'Moderate',
    'Alignment': 'Moderate',
}

hypothetical_evidence = {
    'Grounding': 'Alerts are checked against an explicit security knowledge graph.',
    'Instructibility': 'Analyst feedback can update rules, but model updates require retraining.',
    'Alignment': 'Policy constraints are represented, with limited deployment validation.',
}

hypothetical_record = qualitative_assessment_record(
    system='Hypothetical KG-Enhanced IDS',
    tier='Illustrative only',
    assessments=hypothetical_assessments,
    evidence=hypothetical_evidence,
)

hypothetical_record

## 3. Inspecting Representative Systems from the Survey

We load the categorical assessments corresponding to Table 2 directly from `data/gia_scores.csv`. The colored table preserves the published labels without converting them to numbers, averaging tiers, or ranking systems.

In [ ]:
gia_assessments = pd.read_csv(REPO_ROOT / 'data' / 'gia_scores.csv')
assessment_columns = [
    'system', 'tier', 'grounding_assessment',
    'instructibility_assessment', 'alignment_assessment',
    'key_metric', 'metric_value',
]
display(gia_assessments[assessment_columns])

profile = gia_assessments.set_index('system')[[
    'grounding_assessment',
    'instructibility_assessment',
    'alignment_assessment',
]].copy()
profile.columns = ['Grounding', 'Instructibility', 'Alignment']

category_colors = {
    'Strong': '#C6E0B4',
    'Moderate': '#FFE699',
    'Limited': '#F4B183',
}

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axis('off')
table = ax.table(
    cellText=profile.values,
    rowLabels=profile.index,
    colLabels=profile.columns,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

for row_idx, row in enumerate(profile.itertuples(index=False), start=1):
    for col_idx, label in enumerate(row):
        table[(row_idx, col_idx)].set_facecolor(category_colors[label])

ax.set_title(
    'Representative Qualitative G-I-A Assessments (No Numeric Encoding)',
    fontsize=13,
    fontweight='bold',
    pad=18,
)
plt.tight_layout()
plt.show()

print('These author-assessed categories are illustrative, not measured values or rankings.')

## 4. Survey Corpus Overview

Quick summary statistics from the 108-paper catalog.

In [ ]:
# Load catalog
df = pd.read_csv(REPO_ROOT / 'data' / 'paper_catalog.csv')

print(f"Total papers: {len(df)}")
print(f"\nBy Integration Tier:")
tier_counts = df['tier'].value_counts().sort_index()
tier_names = {'A': 'Deep NeSy', 'B': 'Structured Neural-Symbolic', 'C': 'Contextual Baselines'}
for tier, count in tier_counts.items():
    pct = count / len(df) * 100
    print(f"  Type {tier} ({tier_names[tier]}): {count} papers ({pct:.1f}%)")

print(f"\nBy Subtype (Type B breakdown):")
type_b = df[df['tier'] == 'B']
for subtype, count in type_b['subtype'].value_counts().items():
    print(f"  {subtype}: {count} papers")

print(f"\nBy Year:")
for year, count in df['year'].value_counts().sort_index().items():
    print(f"  {year}: {count} papers")

In [ ]:
# Visualization: Tier distribution and domain coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Tier distribution
colors_tier = ['#2E8B57', '#1F77B4', '#808080']
labels_tier = [f'Type A\nDeep NeSy\n({tier_counts["A"]} papers)',
               f'Type B\nStructured Neural-Symbolic\n({tier_counts["B"]} papers)',
               f'Type C\nBaselines\n({tier_counts["C"]} papers)']
axes[0].pie(tier_counts.values, labels=labels_tier, colors=colors_tier,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Integration Tier Distribution', fontsize=14, fontweight='bold')

# Right: Top domains
domain_counts = df['domain'].value_counts().head(10)
axes[1].barh(range(len(domain_counts)), domain_counts.values, color='#1F77B4', alpha=0.8)
axes[1].set_yticks(range(len(domain_counts)))
axes[1].set_yticklabels(domain_counts.index, fontsize=11)
axes[1].set_xlabel('Number of Papers', fontsize=13)
axes[1].set_title('Top Application Domains', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(REPO_ROOT / 'figures' / 'corpus_overview.pdf', bbox_inches='tight')
plt.savefig(REPO_ROOT / 'figures' / 'corpus_overview.png', bbox_inches='tight', dpi=300)
plt.show()

## Note on Scope

This notebook is a **qualitative walkthrough** of the G-I-A analytical lens. The manuscript's `Consistency`, `Adaptation`, and `Objective` functions remain schematic because:

1. Different cybersecurity domains require different operationalizations
2. The lens is used to structure analysis, not prescribe universal metrics
3. Concrete operationalization and validation remain future research needs

For the full discussion, see Section 2.1 of the paper.